##**Sample-based map accuracy and area estimation**
This notebook demonstrates stratified random sampling for joint map accuracy and area estimation using an Earth Observation (EO) product for stratification.

Author: Josef Wagner ([University of Strasbourg](https://www.unistra.fr/fr), [NASA Harvest](https://www.nasaharvest.org/)) jwagner@unistra.fr

### **Context**

Generating land use / land cover (LULC) maps is becoming increasingly straightforward with end-to-end systems such as the [OLMO Earth platform](https://olmoearth.allenai.org/) or [Google Earth Engine](https://earthengine.google.com/). But every map produced this way still contains classification errors. Traditionally, those errors are assessed by holding out a deterministic subset of the training/reference data (e.g. a train/test split) and reporting an accuracy score on it. That score tells us something about the map, but because the held-out subset is not a probability sample of the region of interest, we have no principled way to know how trustworthy the accuracy estimate itself is.

The same problem shows up when a map is used to derive an area estimate -- of cropland, of forest, of change, or any other class -- by simply counting pixels. Pixel counting on an imperfect map may, by luck, land close to the true area, but without a probability sample there is no way to attach a defensible uncertainty to that number.

[McRoberts, 2011](https://doi.org/10.1016/j.rse.2010.10.013) makes this point forcefully in a paper titled "Satellite image-based maps: Scientific inference or pretty pictures?". His conclusions are as follows:
- A map should not be used without a proper assessment of uncertainties;
- Without such an assessment, a map is just a pretty picture.

Essentially, and against standard practice in the machine learning / deep learning community, a deterministic held-out sample cannot be used to properly assess map accuracy.

This matters most where it is least optional: official crop statistics are routinely required to meet a specified precision. For example, the USDA's June Area Survey (JAS) targets a coefficient of variation (CV) of around 2% for the national wheat area estimate, and Ukraine's State Statistics Service (SSSU) targets a 5% CV for wheat. A number without a defensible uncertainty simply cannot be checked against such a requirement.

[Olofsson et al., 2014](https://www.sciencedirect.com/science/article/abs/pii/S0034425714000704) established the standard pipeline that resolves both problems at once: **probability sampling is the general solution**. With a well-designed sample, we can estimate both map accuracy (rather than a deterministic, held-out accuracy score) and class area, each with a defensible confidence interval -- for any map, anywhere.

Through the example of cropland mapping in the Plateaux region of Togo, this workshop demonstrates how to use a thematic map for joint sample-based area estimation and accuracy assessment. Because sampling normally requires either ground-surveys or photo-interpretation for reference data collection, in this exercise we use a 2019 cropland map from [Kerner & al., 2019](https://arxiv.org/abs/2006.16866) as "true" reference values. A modified version of the map is treated as the inference (i.e. the crop map used by a stakeholder to estimate areas).

This workshop was developed as training material under the FAO EOSTAT Togo initiative, in partnership with the [Togo Data Lab](https://datalab.gouv.tg/) and the [DPSSE](https://planifeducation.gouv.tg/dpsse/). It was first presented at the *Data Infrastructure and Development* event in Lomé, Togo (February 18–19, 2026).

Note that the estimated areas and accuracies obtained in this exercise do not reflect real cropland statistics and are used solely for methodological demonstration purposes.

### **Learning objectives**
Participants will learn the key steps of sample-based map accuracy and area estimation:
- Choosing and defining the sampling design and estimator:
    - Stratified random sampling and design-based estimator
- Planning sample size and allocation for a target precision
- Drawing the sample
- Estimating map accuracy and class areas, with associated uncertainties
- Interpreting results

### **Important resources**

- Full cropland maps for Togo: [Kerner & al., 2019](https://arxiv.org/abs/2006.16866).
- Design-based inference for accuracy assessment and area estimation: [McRoberts, 2011](https://doi.org/10.1016/j.rse.2010.10.013), [Olofsson et al., 2014](https://www.sciencedirect.com/science/article/abs/pii/S0034425714000704)
- Neyman allocation and sample size planning: [Song et al., 2017](https://doi.org/10.1016/j.rse.2017.01.008)


### **Setup**
First, install packages and import helper functions

In [ ]:
## The tricky part: installing gdal

# Update packages
!apt-get update -qq

# Install GDAL system libraries
!apt-get install -y gdal-bin libgdal-dev

import os
os.environ['CPLUS_INCLUDE_PATH'] = '/usr/include/gdal'
os.environ['C_INCLUDE_PATH'] = '/usr/include/gdal'

%pip install --upgrade pip -q
%pip install numpy pandas shapely fiona geopandas rasterio -q
# Optional: install GDAL Python bindings (match the system version)
%pip install gdal==$(gdal-config --version) -q

from osgeo import gdal, ogr, osr

print("GDAL version:", gdal.__version__)

## The rest is easy
%pip install pandas geopandas shapely numpy matplotlib folium -q

In [ ]:
def in_colab():
  try:
    import google.colab
    return True
  except ImportError:
    return False

if in_colab():
  !git clone https://github.com/jowa-ea/Class_EO-area-accuracy-sampling.git
  %cd Class_EO-area-accuracy-sampling
else:
  print("Running locally - skipping git clone")

In [ ]:
# Import packages
import os
from osgeo import gdal
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
pd.options.display.float_format = '{:.2f}'.format
gdal.UseExceptions()

# Import modules
import utils_stratified_random_sampling as srs
import utils_plotting as plot

### **Available Data**

We have two cropland maps of the Plateaux region of Togo:

- `crop_map_true` – reference map showing the "true" cropland areas (left).  
- `crop_map_pred` – predicted cropland map (right).  

Both maps use the same colors:  
- Grey (0): non-cropland  
- Green (1): cropland  

Our goal is to estimate the cropland area in this region.


> **Q1. Compare the two maps. What do you notice, and what are the consequences?**

>When looking at the maps, we see that the predicted map overestimates cropland compared to the reference map.  

>- Simply counting green pixels in the predicted map would overestimate the actual cropland area.  
>- In reality, we never have a perfect reference map, so the exact cropland area is unknown.  

>**Takeaway:** To estimate area accurately from a map, we need to rely on **sampling and statistical methods** rather than just pixel counting.



![Cropland maps](https://github.com/jowa-ea/EO4AreaEstimates/blob/a74ca57ddb98bac8bd4a00bc15f43ea0df40a138/images/DID_cropmaps.png?raw=true)

### **Stratified random sampling for area estimation**








In [ ]:
## Paths setup
if in_colab():
    demo_data_path = '/content/Class_EO-area-accuracy-sampling/demo_data'
else:
    demo_data_path = './demo_data'
crop_map_true = os.path.join(demo_data_path, 'CropMapTrue.tif')
crop_map_pred = os.path.join(demo_data_path, 'CropMapPred.tif')

# Create output directory for stratified random sampling
srs_outdir = demo_data_path.replace('demo_data','stratifiedRandomSampling_outputs')
os.makedirs(srs_outdir, exist_ok=True)

#### **Step 1.** Pixel counting areas from inference map classes

>*In sampling the correct terminology for a subdivision of the population is a **stratum**. Therefore, the inference map is a population of pixels, that are divided into strata **cropland** and **non-cropland***.

Here the user needs to specify the strata of interest in shape of a list.

In [ ]:
# Defining strata of interest
strata = [0,1] # 0 = non-cropland, 1 = cropland

In [ ]:
# Pixel counting area, saving as csv for later, displaying results
output_csv = os.path.join(srs_outdir, 'pixelcounts_predicted_map.csv')
pixel_counts_pred = srs.compute_pixel_counts(crop_map_pred, strata = strata, output_csv=output_csv) #output in pixels
df = pd.read_csv(output_csv)
df

In the previous dataframe, the pixel counted area is expressed in hectares, *Wi* represents the proportion of the total area covered by each stratum.

#### **Step 2.** Sample size and allocation

>*When using a map for sampling, the sample unit is defined as one pixel. The sample size, noted Ni, is the total number of sample units to be drawn. The sample size depends on (i) targeted accuracy of area estimates, (ii) available resources. Note. If accuracy levels are not satisfactory, in stratified random sampling, it is straightforward to add more samples to reduce uncertainties.*

The objective of the sample allocation is to make sure each stratum is correctly represented in the sample, when defining the number of sample units allocated to each stratum *h* (noted *nh*), and that the total sample size *n_tot* is large enough to achieve a target precision.

Rather than picking *n_tot* arbitrarily, we determine it from a target **coefficient of variation (CV)** on the cropland area estimate, at a given **confidence level** (e.g. 95%), using **Neyman allocation** (Cochran, 1977; see also Song et al., 2017):

$$n_{tot} = \frac{z^2 \left(\sum_h W_h S_h\right)^2}{E^2} \qquad\qquad n_h = n_{tot}\frac{W_h S_h}{\sum_h W_h S_h}$$

where *Wh* is the stratum weight, *Sh* is the within-stratum standard deviation, *z* is the standard normal deviate for the chosen confidence level, and *E = cv_target &times; p_target* is the desired confidence-interval half-width on the target class's proportion.

>*Because E already scales with z, requiring E &le; cv_target&middot;p_target is equivalent to requiring the estimator's coefficient of variation, SE(p&#770;)/p_target, to be at most cv_target/z. In other words, for the same cv_target, a higher confidence level (larger z) requires a larger sample.*

*Sh* is not known before sampling. We get a prior estimate of it from a small **deterministic pilot sample**: a stratified random sample of 100 sample units drawn with proportional allocation, for which we recover the reference class and compute each stratum's user's accuracy *Uh*. Since *Sh = sqrt(Uh (1 - Uh))* (Olofsson et al., 2014), this pilot sample gives us the prior variance estimate we need for the Neyman formulas above.

In [ ]:
# Step 2a. Deterministic pilot sample (100 SU, proportional allocation) to estimate prior stratum variances (Sh)
pilot_Ni = 100  # pilot sample size
pilot_min_allocation = 0  # pure proportional allocation for the pilot
pilot_allocation = srs.allocate_samples(pixel_counts_pred, min_allocation=pilot_min_allocation, Ni=pilot_Ni)

pilot_samples_pred_gpkg = os.path.join(srs_outdir, 'pilot_sample.gpkg')
pilot_samples_pred = srs.draw_samples(crop_map_pred, pilot_allocation, pilot_samples_pred_gpkg, seed=0)

pilot_samples_true_gpkg = os.path.join(srs_outdir, 'pilot_sample_annotated.gpkg')
pilot_samples_true = srs.extract_true_class_values(pilot_samples_pred_gpkg, crop_map_true, pilot_samples_true_gpkg, nodata_class=0)

Sh = srs.compute_pilot_variances(strata, list(pilot_samples_true.stratum_pred), list(pilot_samples_true.true_class), v=True)
Sh

In [ ]:
# Step 2b. Total sample size (Neyman) for a target confidence level and target coefficient of variation
cv_target = 0.04    # target relative precision on the cropland area estimate (e.g. 4%)
confidence = 0.95   # confidence level (e.g. 95%)
target_stratum = 1  # cropland

n_tot = srs.compute_neyman_sample_size(pixel_counts_pred, Sh, cv_target=cv_target, confidence=confidence, target_stratum=target_stratum, v=True)

allocation_csv_path = os.path.join(srs_outdir, 'stratum_allocations.csv')
allocation = srs.allocate_neyman(pixel_counts_pred, Sh, n_tot, output_allocation_csv=allocation_csv_path, v=True)
df = pd.read_csv(allocation_csv_path)
df

#### **Step 3.** Drawing the samples from the map

In [ ]:
# Step 3. Draw samples from a map
samples_pred_gpkg = os.path.join(srs_outdir,'stratified_random_sample.gpkg')
samples_pred = srs.draw_samples(crop_map_pred,allocation, samples_pred_gpkg, seed = 0,v=True)
# Here also display as a map visual
samples_pred.head(10)

In [ ]:
m = plot.plot_togo_layers_geom(
    layers = [samples_pred],
    color_col_list = ['stratum_pred'],
    palette_list = [{0:"grey", 1:"blue"}],
    legend_name_list = ['Pred strata'],
    zoom_start = 9,
    map_height = "700px",
    map_width = "50%"
)
m

> **Q2. Looking at the map above, what do you notice regarding the spatial distribution of sample units? What are the consequences?**

>- Due to random sampling of sample units within strata, the sample units are spatially scattered all across the study region.
>- Visiting each sample unit on the ground would be highly inefficient and time consuming.

>**Takeaway:** when using stratified random sampling as a sampling design, sample units are generally **annotated with remote sensing imagery interpretation** rather than visited on the ground.

#### **Step 4.** Reference data collection
Here we subsitute the remote sensing imagery interpretation for reference data collection with direct recovery of "true" values from the reference map.



In [ ]:
# Step 4. Recovering true values from reference map
samples_true_gpkg = os.path.join(srs_outdir, 'stratified_random_sample_annotated.gpkg')
nodata_class = 0 # If the true value is nodata, value to be attributed instead (here all no-data go to non-crop)
samples_true = srs.extract_true_class_values(samples_pred_gpkg,crop_map_true,samples_true_gpkg, nodata_class, v=True)
samples_true.head(10)

In the next cell you are generating a map of predicted vs. reference values. You can use the layer control (top right of the map) to toggle layers.

In [ ]:
# Plotting on map
m = plot.plot_togo_layers_geom(
    layers = [samples_pred, samples_true],
    color_col_list = ['stratum_pred', 'true_class'],
    palette_list = [{0:"grey", 1:"blue"},{0:"grey", 1:"blue"}],
    legend_name_list = ['Predicted', 'reference'],
    zoom_start = 9,
    map_height = "700px",
    map_width = "50%"
)
m

#### **Step 5.** Crop area and uncertainty estimation

- The area, standared error (SE) and 95% confidence interval (CI) are expressed in Ha.
- The coefficient of variation (SE% = SE / Area) is expressed as a proportion (1 = 100%)
- The relative confidence interval (CI% = CI / Area) is expressed as a proportion.

In [ ]:
# Step 5. Compute sample-based area estimates and uncertainties
pred, true = list(samples_true.stratum_pred), list(samples_true.true_class)
metrics_csv_path = os.path.join(srs_outdir, 'stratified_random_sample_metrics.csv')
metrics = srs.compute_stratified_random_sampling_metrics(pixel_counts_pred,pred,true, output_csv=metrics_csv_path)
print('Estimates areas and uncertainties with stratified random sampling')
metrics

Beyond area, the same sample also lets us assess the **accuracy of the map** -- how well the predicted map agrees with the reference data. Following Olofsson et al. (2014), we estimate:

- **Overall accuracy (OA)**: the proportion of the area correctly classified.
- **User's accuracy (UA)** of a class: the proportion of the area mapped as that class that is actually that class in the reference data (1 - UA = commission error).
- **Producer's accuracy (PA)** of a class: the proportion of the area that is actually that class in the reference data that was mapped as that class (1 - PA = omission error).

Each estimate comes with a standard error and 95% confidence interval, computed from the same stratified sample used for area estimation.

In [ ]:
# Compute map accuracy (overall, user's, producer's) with uncertainties
accuracy_csv_path = os.path.join(srs_outdir, 'accuracy_metrics.csv')
accuracy_metrics, overall_accuracy = srs.compute_accuracy_metrics(pixel_counts_pred, pred, true, output_csv=accuracy_csv_path)
print(f"Overall accuracy: {overall_accuracy['O']:.3f} +/- {overall_accuracy['CI']:.3f}")
accuracy_metrics

Comparison to pixel counted areas

In [ ]:
print("Pixel counting estimator areas")
pixel_counts_pred

> **Q3. Suggested exercise: test different sample sizes (Ni) and observe how uncertainties change, what conclusions can you draw?**

>- Augmenting the sample size will reduce the uncertainty around area estimates.
>- As sample size increases, the reduction in uncertainty is **not linear** but approximately follows a **square-root relationship** (`SE ∝ 1/√Ni`). This effect is particularly pronounced for **rare classes** representing less than ~5% of the study area: initially, increasing the number of samples efficiently reduces uncertainty, but the relationship is **asymptotic**—beyond a certain point, each additional sample contributes very little, and achieving even a 1% further reduction in uncertainty may require hundreds more samples. In other words, while the standard error decreases with `√Ni`, the benefits of additional sampling **diminish rapidly for rare classes**, highlighting the trade-off between sampling effort and precision.


#### **Conclusion stratified random sampling**
**Pro's:**
- Efficient sampling design, small sample sizes allow to efficiently reduce uncertainties.
- Time and resource efficient with no ground visits required, the full workflow can be run from behind a computer.
- No ground visits required: ideal when conflict, weather events or distance to study region prevents ground access

**Con's:**
- Not adapted for ground visits
- Remote sensing imagery interpretation requires dedicated software and domain expertise. Here is an example of a Google Earth Engine [point annotation tool](https://github.com/jowa-ea/Sentinel-2_PointAnnotationTool.git) for annotation sample units. The example code runs for Ukraine, but can be adapted to other geographies.

---
